In [74]:
from sqlalchemy import create_engine, text
import os
import pandas as pd
import numpy as np

DB_URL = os.getenv("SUPABASE_DB_URL")  # postgres://...

engine = create_engine(DB_URL)

df_match = pd.read_sql(
    """
    SELECT
        "matchId",
        "teamId",
        "xG_model",
        "h_a"
    FROM eredivisie_events
    WHERE "xG_model" IS NOT NULL AND "xG_model" <>0
    """,
    engine
)



In [75]:
from sqlalchemy import create_engine
import os
import pandas as pd
import numpy as np
from collections import Counter

# =========================
# 2) Monte Carlo helpers
# =========================
def simulate_team_goals(xg_values, n_sim=10000, rng=None):
    rng = rng or np.random.default_rng()
    xg = np.asarray(xg_values, dtype=float)
    if xg.size == 0:
        return np.zeros(n_sim, dtype=int)
    return rng.binomial(1, xg, size=(n_sim, xg.size)).sum(axis=1)

def simulate_match_from_h_a(match_events, n_sim=20000, h_label="h", a_label="a", seed=42):
    rng = np.random.default_rng(seed)

    xg_h = match_events.loc[match_events["h_a"] == h_label, "xG_model"].values
    xg_a = match_events.loc[match_events["h_a"] == a_label, "xG_model"].values

    home_goals = simulate_team_goals(xg_h, n_sim=n_sim, rng=rng)
    away_goals = simulate_team_goals(xg_a, n_sim=n_sim, rng=rng)

    return home_goals, away_goals

def most_likely_score_and_prob(home_goals, away_goals):
    scorelines = list(zip(home_goals, away_goals))
    (h, a), cnt = Counter(scorelines).most_common(1)[0]
    prob = cnt / len(scorelines)
    return int(h), int(a), float(prob)

def points_from_score(h, a):
    if h > a:
        return 3, 0
    if h < a:
        return 0, 3
    return 1, 1

# =========================
# 3) Run per match
# =========================
rows = []

for match_id, match_events in df.groupby("matchId", sort=False):

    # ---- teamIds ophalen ----
    home_team_ids = match_events.loc[match_events["h_a"] == "h", "teamId"].unique()
    away_team_ids = match_events.loc[match_events["h_a"] == "a", "teamId"].unique()

    # safety checks
    home_team_id = int(home_team_ids[0]) if len(home_team_ids) > 0 else None
    away_team_id = int(away_team_ids[0]) if len(away_team_ids) > 0 else None

    # ---- Monte Carlo ----
    home_goals, away_goals = simulate_match_from_h_a(
        match_events,
        n_sim=20000,
        h_label="h",
        a_label="a",
        seed=42,
    )

    h_sim, a_sim, score_prob = most_likely_score_and_prob(home_goals, away_goals)
    home_points, away_points = points_from_score(h_sim, a_sim)

    p_home_win = float(np.mean(home_goals > away_goals))
    p_draw     = float(np.mean(home_goals == away_goals))
    p_away_win = float(np.mean(home_goals < away_goals))

    exp_home_points = float(3 * p_home_win + 1 * p_draw)
    exp_away_points = float(3 * p_away_win + 1 * p_draw)

    rows.append({
        "matchId": int(match_id),

        # team identifiers
        "home_teamId": home_team_id,
        "away_teamId": away_team_id,

        # simulated most-likely score
        "home_goals_sim": h_sim,
        "away_goals_sim": a_sim,
        "score_prob": score_prob,

        # points from that score
        "home_points": home_points,
        "away_points": away_points,

        # W/D/L probabilities
        "p_home_win": p_home_win,
        "p_draw": p_draw,
        "p_away_win": p_away_win,

        # expected points
        "exp_home_points": exp_home_points,
        "exp_away_points": exp_away_points,
    })

sim_results_df = pd.DataFrame(rows)

# =========================
# 4) sanity check
# =========================
print("unique matches:", df["matchId"].nunique())
print("rows in output:", len(sim_results_df))
print(sim_results_df.head())



unique matches: 198
rows in output: 198
   matchId  home_teamId  away_teamId  home_goals_sim  away_goals_sim  \
0  1903835          113          867               2               1   
1  1903831          129          783               2               2   
2  1903812          130          762               5               0   
3  1903825          758          242               3               1   
4  1903814          256          870               2               2   

   score_prob  home_points  away_points  p_home_win   p_draw  p_away_win  \
0     0.10485            3            0     0.71295  0.16490     0.12215   
1     0.08715            1            1     0.61725  0.19350     0.18925   
2     0.13430            3            0     0.98710  0.01095     0.00195   
3     0.09870            3            0     0.86960  0.08650     0.04390   
4     0.11180            1            1     0.45410  0.24810     0.29780   

   exp_home_points  exp_away_points  
0          2.30375          0.53

In [76]:
sim_results_df.to_sql(
    "monte carlo",
    engine,
    if_exists="replace",     # eerste keer: "replace"
    index=False,
    method="multi",
    chunksize=1000
)

198